<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Dataset-3-Preprocessing/Group_12_Dataset_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1: Importing Packages

In [3]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [4]:
#MUST BE RUN inorder for file download to work
!pip install gdown

#Downloading dataset csv into runtime from google drive link
import gdown

#train dataset
url = "https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt"
gdown.download(url, "train.csv", quiet=False)

#test dataset
url = "https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV"
gdown.download(url, "test.csv", quiet=False)

#loading csv files into pandas dataframes
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

Downloading...
From (original): https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt
From (redirected): https://drive.google.com/uc?id=1WxiA2-DJuTOcykkqqTX-LECs9XltPHXt&confirm=t&uuid=d8ab0df5-336f-4b03-951c-da762797bf76
To: /content/train.csv
100%|██████████| 351M/351M [00:02<00:00, 130MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV
From (redirected): https://drive.google.com/uc?id=10qbFPqH3JYWfGmDSaal7IGzfGLb86HQV&confirm=t&uuid=d9a32811-91e6-4ce5-8116-1a87cd20eeef
To: /content/test.csv
100%|██████████| 150M/150M [00:01<00:00, 92.7MB/s]


In [5]:
#printing dataset information (how many rows, columns, datatype etc)
print(train_df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(train_df.isnull().sum())
print('----------------------------------------------------------')

# Show ONLY columns with missing values
print("Columns with missing values:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print('----------------------------------------------------------')

#Dropping rows with missing values, if any
train_df = train_df.dropna()

#Verifying that those rows have been dropped and there are no missing values
print("Total missing values after dropping:")
print(train_df.isnull().sum().sum())  # should be 0
print('----------------------------------------------------------')

#printing first 5 rows of data to ensure it correlates with datafile
print(train_df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(train_df['is_fraud'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

##Step 3: Dropping Unnecessary Fields


In [6]:
#Personal identifiers such as first and last names are irrelevant in prediction and hence removed
#trans_num is also a unique identifier for each transaction hence no patterns can be derived hence it is removed
#The unnamed column contained indexes for each entry in the dataset and is irrelevant for prediction
#Features such as street, city, state and zip could have been kept however due to the presence of numerous more important features
# it was removed to due to high cardinality and would cause an explosion into too many categories.

train_df = train_df.drop(columns=[
    'Unnamed: 0',
    'first',
    'last',
    'street',
    'city',
    'state',
    'zip',
    'trans_num'
])

In [7]:
#Viewing data after dropping unnecessary columns
print(train_df.head(5))

  trans_date_trans_time            cc_num                            merchant  \
0   2019-01-01 00:00:18  2703186189652095          fraud_Rippin, Kub and Mann   
1   2019-01-01 00:00:44      630423337322     fraud_Heller, Gutmann and Zieme   
2   2019-01-01 00:00:51    38859492057661                fraud_Lind-Buckridge   
3   2019-01-01 00:01:16  3534093764340240  fraud_Kutch, Hermiston and Farrell   
4   2019-01-01 00:03:06   375534208663984                 fraud_Keeling-Crist   

        category     amt gender      lat      long  city_pop  \
0       misc_net    4.97      F  36.0788  -81.1781      3495   
1    grocery_pos  107.23      F  48.8878 -118.2105       149   
2  entertainment  220.11      M  42.1808 -112.2620      4154   
3  gas_transport   45.00      M  46.2306 -112.1138      1939   
4       misc_pos   41.96      M  38.4207  -79.4629        99   

                                 job         dob   unix_time  merch_lat  \
0          Psychologist, counselling  1988-03-09  132

##Step 4: Feature Engineering


In [8]:
#Converting fields with date & time to appropriate datetime type
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['dob'] = pd.to_datetime(train_df['dob'])

In [9]:
#Extracting individual features from trans_date_trans_time feature
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour
train_df['day'] = train_df['trans_date_trans_time'].dt.day
train_df['month'] = train_df['trans_date_trans_time'].dt.month
train_df['dayofweek'] = train_df['trans_date_trans_time'].dt.dayofweek

In [10]:
#Converting DOB to age
train_df['age'] = (train_df['trans_date_trans_time'] - train_df['dob']).dt.days

In [11]:
#Dropping original trans_date_trans_time and dob columns
train_df = train_df.drop(columns=['trans_date_trans_time', 'dob'])

In [12]:
#Encoding categorical values using OneHotEncoder
train_df = pd.get_dummies(train_df, columns=['gender', 'merchant', 'category', 'job'], drop_first=True)

##Step 5: Splitting features & target

In [13]:
#Splitting features & target
X = train_df.drop('is_fraud', axis=1)
y = train_df['is_fraud']

X_test_final = test_df.drop('is_fraud', axis=1)
y_test_final = test_df['is_fraud']